# Multi-Agent HS Teacher Assistant



In [1]:
# Cell 1: installs
!pip -q install openai faiss-cpu sentence-transformers pypdf numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 31.4 MB/s eta 0:00:00


In [2]:
# Cell 2: mount drive + API key
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ["OPENAI_API_KEY"], "Missing OPENAI_API_KEY in Colab Secrets"

Mounted at /content/drive


## GPT Version - one we should actually use

In [3]:


# Cell 3: config
import os

# Update these paths if needed
ROOT_DIR = "/content/drive/MyDrive/Advanced AI/Midterm/midterm_data_clean"

# Use the indexes built from your v3 notebook
INDEX_DIR = "/content/drive/MyDrive/Advanced AI/Midterm/indexes_hs_teacher_clean"

# Embeddings should match the original FAISS index construction
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# OpenAI model for routing / rewrite / rerank / answering
OPENAI_MODEL = "gpt-5"

# Retrieval defaults
FETCH_K = 50
FINAL_K = 6

print("ROOT_DIR:", ROOT_DIR)
print("INDEX_DIR:", INDEX_DIR)
print("EMBED_MODEL_NAME:", EMBED_MODEL_NAME)
print("OPENAI_MODEL:", OPENAI_MODEL)

ROOT_DIR: /content/drive/MyDrive/Advanced AI/Midterm/midterm_data_clean
INDEX_DIR: /content/drive/MyDrive/Advanced AI/Midterm/indexes_hs_teacher_clean
EMBED_MODEL_NAME: sentence-transformers/all-MiniLM-L6-v2
OPENAI_MODEL: gpt-5


In [4]:
# Cell 4: imports + clients
import os, re, json, textwrap
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import faiss
from openai import OpenAI
from sentence_transformers import SentenceTransformer

client = OpenAI()
st_model = SentenceTransformer(EMBED_MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# Cell 5: load indexes
def load_index(index_dir: str, name: str):
    index_path = os.path.join(index_dir, f"{name}.faiss")
    meta_path = os.path.join(index_dir, f"{name}.meta.jsonl")

    if not os.path.exists(index_path):
        raise FileNotFoundError(f"Missing index file: {index_path}")
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"Missing metadata file: {meta_path}")

    index = faiss.read_index(index_path)
    meta = []
    with open(meta_path, "r", encoding="utf-8") as f:
        for line in f:
            meta.append(json.loads(line))
    return index, meta

INDEX_NAMES = [
    "standards_core",
    "curriculum_overview",
    "exam_questions",
    "exam_scoring",
]

INDEXES = {name: load_index(INDEX_DIR, name) for name in INDEX_NAMES}

for name, (idx, meta) in INDEXES.items():
    print(f"{name}: dim={idx.d}, vectors={idx.ntotal}, meta_rows={len(meta)}")

standards_core: dim=384, vectors=0, meta_rows=0
curriculum_overview: dim=384, vectors=4506, meta_rows=4506
exam_questions: dim=384, vectors=1676, meta_rows=1676
exam_scoring: dim=384, vectors=3147, meta_rows=3147


In [6]:
# Cell 6: embedding + retrieval helpers
def embed_384(texts: List[str]) -> np.ndarray:
    vecs = st_model.encode(texts, normalize_embeddings=True)
    return np.asarray(vecs, dtype=np.float32)

def infer_subject_from_query(query: str) -> Optional[str]:
    q = query.lower()
    mapping = {
        "algebra i": "Algebra 1",
        "algebra 1": "Algebra 1",
        "algebra ii": "Algebra 2",
        "algebra 2": "Algebra 2",
        "geometry": "Geometry",
        "ela": "High School English Language Arts",
        "english language arts": "High School English Language Arts",
        "english": "High School English Language Arts",
        "chemistry": "Chemistry",
        "physics": "Physics",
        "living environment": "Living Environment",
        "biology": "Life Science: Biology",
        "global history": "Global History and Geography II",
    }
    for k, v in mapping.items():
        if k in q:
            return v
    return None

def infer_admin_from_query(query: str) -> Optional[str]:
    q = query.lower()
    patterns = [
        (r"january\s+20\d{2}", None),
        (r"august\s+20\d{2}", None),
        (r"june\s+20\d{2}", None),
    ]
    for pat, _ in patterns:
        m = re.search(pat, q)
        if m:
            return m.group(0).title()
    return None

QUESTIONISH_PATTERNS = [
    r"\bPart I\b",
    r"\bQuestion\b",
    r"\([0-9]+\)",
    r"\b[A-D]\)",
    r"(?m)^\s*[0-9]{1,2}[\.)]\s",
]

def looks_like_question_text(text: str) -> bool:
    t = (text or "").replace("\n", " ")
    return any(re.search(p, t) for p in QUESTIONISH_PATTERNS)

def is_front_matter_exam(row: Dict[str, Any]) -> bool:
    src = os.path.basename(row.get("source", "")).lower()
    page = row.get("page")
    if "exam" in src and isinstance(page, int) and page <= 1:
        return not looks_like_question_text(row.get("text", ""))
    return False

def module_hint(query: str) -> Optional[str]:
    m = re.search(r"\bmodule\s*(\d+)\b", query.lower())
    return m.group(1) if m else None

def retrieve_faiss(query: str, index_name: str, fetch_k: int = FETCH_K, final_k: int = FINAL_K) -> List[Dict[str, Any]]:
    index, meta = INDEXES[index_name]
    qv = embed_384([query])
    fetch_k = min(fetch_k, len(meta))
    scores, ids = index.search(qv, fetch_k)

    subj = infer_subject_from_query(query)
    admin = infer_admin_from_query(query)
    mod = module_hint(query)

    cands = []
    for sc, idx in zip(scores[0].tolist(), ids[0].tolist()):
        if idx < 0 or idx >= len(meta):
            continue
        row = dict(meta[idx])
        row["score"] = float(sc)
        cands.append(row)

    def boosted_score(r: Dict[str, Any]) -> float:
        score = r["score"]
        if subj and r.get("subject") == subj:
            score += 0.15
        if admin and r.get("admin") == admin:
            score += 0.10
        if index_name == "curriculum_overview" and mod:
            src = (r.get("source") or "").lower().replace("\\", "/")
            base = os.path.basename(src)
            if f"-m{mod}-" in base or f"/module {mod}/" in src:
                score += 0.25
        if index_name == "exam_questions" and r.get("doc_type") == "exam":
            score += 0.10
        return score

    cands.sort(key=boosted_score, reverse=True)

    if index_name == "exam_questions":
        cands = [r for r in cands if not is_front_matter_exam(r)]
        cands.sort(
            key=lambda r: (looks_like_question_text(r.get("text", "")), boosted_score(r)),
            reverse=True,
        )

    out = []
    seen = set()
    for r in cands:
        if r["doc_id"] in seen:
            continue
        seen.add(r["doc_id"])
        out.append(r)
        if len(out) >= final_k:
            break
    return out

In [7]:
# Cell 7: GPT retrieval helpers (from the gptv4-2 style)
def rewrite_queries(user_query: str, agent_name: str, index_names: List[str], model: str = OPENAI_MODEL) -> List[str]:
    prompt = f'''
Rewrite the user request into 4 short search queries for PDF retrieval.

Agent: {agent_name}
Candidate indexes: {", ".join(index_names)}

Rules:
- Return exactly 4 lines
- No numbering
- Each line <= 14 words
- Preserve subject / grade / exam / module details when present
- Prefer retrieval-oriented wording

User request: {user_query}
'''.strip()

    resp = client.responses.create(
        model=model,
        input=prompt,
        reasoning={"effort": "low"},
    )
    lines = [x.strip() for x in resp.output_text.splitlines() if x.strip()]
    if len(lines) >= 4:
        return lines[:4]
    return [user_query] * 4

def rerank_with_gpt(user_query: str, candidates: List[Dict[str, Any]], top_n: int = FINAL_K, model: str = OPENAI_MODEL) -> List[Dict[str, Any]]:
    packed = []
    for c in candidates[:60]:
        packed.append({
            "doc_id": c.get("doc_id"),
            "pdf": os.path.basename(c.get("source", "")),
            "page": c.get("page"),
            "subject": c.get("subject"),
            "admin": c.get("admin"),
            "snippet": (c.get("text", "")[:280] or "").replace("\n", " "),
        })

    prompt = f'''
You are selecting the best evidence chunks for answering a teacher-assistant query.

Return ONLY a JSON array of doc_id strings from best to worst.
Choose at most {top_n} ids.

User query:
{user_query}

Candidates:
{json.dumps(packed, ensure_ascii=False)}
'''.strip()

    resp = client.responses.create(
        model=model,
        input=prompt,
        reasoning={"effort": "low"},
    )

    try:
        chosen_ids = json.loads(resp.output_text.strip())
        chosen_ids = [x for x in chosen_ids if isinstance(x, str)]
    except Exception:
        chosen_ids = []

    by_id = {c["doc_id"]: c for c in candidates}
    reranked = [by_id[d] for d in chosen_ids if d in by_id]
    return reranked[:top_n] if reranked else candidates[:top_n]

In [8]:
# Cell 8: agent specifications
AGENT_SPECS = {
    "regents_agent": {
        "description": "Handles Regents-style practice questions, exam questions, scoring guides, rubrics, and answer explanations.",
        "indexes": ["exam_questions", "exam_scoring"],
    },
    "curriculum_agent": {
        "description": "Handles NYS curriculum teaching support, standards alignment, module guidance, lesson framing, and teaching explanations.",
        "indexes": ["standards_core", "curriculum_overview"],
    },
    "study_skills_agent": {
        "description": "Handles time management, study habits, and college readiness. Not implemented in this notebook.",
        "indexes": [],
    },
}

def keyword_router_fallback(query: str) -> str:
    q = query.lower()

    regents_terms = [
        "regents", "practice question", "practice problems", "multiple choice",
        "constructed response", "rubric", "rating guide", "full credit",
        "scoring", "model response", "exam"
    ]
    curriculum_terms = [
        "standard", "standards", "curriculum", "module", "lesson",
        "teach", "explain", "how do i teach", "nys", "next generation",
        "ela", "algebra", "geometry"
    ]
    study_terms = [
        "time management", "study plan", "organization", "planner",
        "college essay", "college application", "study skills"
    ]

    if any(t in q for t in regents_terms):
        return "regents_agent"
    if any(t in q for t in study_terms):
        return "study_skills_agent"
    if any(t in q for t in curriculum_terms):
        return "curriculum_agent"
    return "curriculum_agent"

def route_with_llm(query: str, model: str = OPENAI_MODEL) -> Dict[str, Any]:
    prompt = f'''
Route the user query to exactly one agent.

Available agents:
- regents_agent: Regents-style practice questions, exam questions, scoring guides, rubrics, answer explanations
- curriculum_agent: NYS curriculum teaching, standards, modules, lessons, concept explanations
- study_skills_agent: time management, study skills, college readiness (not implemented)

Return ONLY valid JSON with keys:
- agent
- rationale
- confidence

User query:
{query}
'''.strip()

    try:
        resp = client.responses.create(
            model=model,
            input=prompt,
            reasoning={"effort": "low"},
        )
        data = json.loads(resp.output_text.strip())
        if data.get("agent") not in AGENT_SPECS:
            raise ValueError("Bad agent")
        return data
    except Exception:
        fallback = keyword_router_fallback(query)
        return {
            "agent": fallback,
            "rationale": "Fallback keyword router used because LLM router output was unavailable.",
            "confidence": 0.4,
        }

In [9]:
# Cell 9: agent retrieval pipeline
def gather_agent_evidence(
    user_query: str,
    agent_name: str,
    fetch_k_per_query: int = FETCH_K,
    final_k: int = FINAL_K,
    model: str = OPENAI_MODEL,
) -> Dict[str, Any]:
    spec = AGENT_SPECS[agent_name]
    index_names = spec["indexes"]

    if not index_names:
        return {
            "rewrites": [],
            "candidate_pool": [],
            "top_evidence": [],
        }

    rewrites = rewrite_queries(user_query, agent_name, index_names, model=model)

    pool = []
    seen = set()

    # search across all indexes assigned to the agent
    per_index_final = max(final_k, 4)
    for rq in rewrites:
        for idx_name in index_names:
            hits = retrieve_faiss(
                rq,
                index_name=idx_name,
                fetch_k=fetch_k_per_query,
                final_k=per_index_final,
            )
            for h in hits:
                if h["doc_id"] not in seen:
                    seen.add(h["doc_id"])
                    h["retrieved_from"] = idx_name
                    pool.append(h)

    # lightweight score sort before GPT rerank
    pool.sort(key=lambda x: x.get("score", 0.0), reverse=True)
    pool = pool[: min(len(pool), max(25, final_k * 5))]

    top = rerank_with_gpt(user_query, pool, top_n=final_k, model=model) if pool else []

    return {
        "rewrites": rewrites,
        "candidate_pool": pool,
        "top_evidence": top,
    }

In [10]:
# Cell 10: answer-generation prompts for each implemented agent
def format_context(evidence: List[Dict[str, Any]]) -> str:
    if not evidence:
        return "NO EVIDENCE FOUND."
    chunks = []
    for i, e in enumerate(evidence, start=1):
        chunks.append(
            f'''CHUNK {i}
DOC_ID: {e["doc_id"]}
INDEX: {e.get("retrieved_from")}
PDF: {os.path.basename(e.get("source", ""))}
PAGE: {e.get("page")}
SUBJECT: {e.get("subject")}
ADMIN: {e.get("admin")}
TEXT:
{e.get("text", "")}'''
        )
    return "\n\n".join(chunks)

def answer_with_regents_agent(user_query: str, evidence: List[Dict[str, Any]], model: str = OPENAI_MODEL) -> str:
    context = format_context(evidence)
    prompt = f'''
You are the Regents Agent for a high school teacher assistant.

Your job:
- Answer Regents-related questions
- Create Regents-style practice only when the user asks for practice
- Use ONLY the evidence when making factual claims about real NYS exams, scoring guides, or released materials
- If the evidence is insufficient, say so clearly
- Never claim a generated question is an official released Regents question unless the evidence shows that

Output style:
- Clear teacher-friendly answer
- Use short sections or bullets when helpful
- End with a brief "Sources used" line listing DOC_IDs

User request:
{user_query}

Evidence:
{context}
'''.strip()

    resp = client.responses.create(
        model=model,
        input=prompt,
        reasoning={"effort": "low"},
    )
    return resp.output_text.strip()

def answer_with_curriculum_agent(user_query: str, evidence: List[Dict[str, Any]], model: str = OPENAI_MODEL) -> str:
    context = format_context(evidence)
    prompt = f'''
You are the Curriculum Agent for a high school teacher assistant.

Your job:
- Help teach using NYS math / ELA curriculum and standards
- Use ONLY the evidence for factual claims about curriculum structure, modules, standards, or official materials
- If the user asks for a teaching explanation, you may explain in your own words, but keep it aligned to the evidence
- If evidence is weak or incomplete, say what you can and what is missing

Output style:
- Practical and teacher-friendly
- Prioritize teaching steps, alignment, lesson framing, and student-facing explanation when relevant
- End with a brief "Sources used" line listing DOC_IDs

User request:
{user_query}

Evidence:
{context}
'''.strip()

    resp = client.responses.create(
        model=model,
        input=prompt,
        reasoning={"effort": "low"},
    )
    return resp.output_text.strip()

def answer_with_unimplemented_agent(user_query: str) -> str:
    return (
        "This query routed to the study-skills / time-management agent, "
        "but that agent is not implemented in this notebook yet."
    )

In [11]:
# Cell 11: orchestration function
def ask_multi_agent(
    user_query: str,
    routing_mode: str = "llm",   # options: "llm" or "manual"
    forced_agent: Optional[str] = None,
    fetch_k_per_query: int = FETCH_K,
    final_k: int = FINAL_K,
    model: str = OPENAI_MODEL,
    show_debug: bool = False,
) -> Dict[str, Any]:
    if forced_agent is not None:
        chosen_agent = forced_agent
        route_info = {
            "agent": forced_agent,
            "rationale": "Agent was manually forced by the caller.",
            "confidence": 1.0,
        }
    elif routing_mode == "manual":
        raise ValueError("For manual mode, pass forced_agent explicitly.")
    else:
        route_info = route_with_llm(user_query, model=model)
        chosen_agent = route_info["agent"]

    retrieval = gather_agent_evidence(
        user_query=user_query,
        agent_name=chosen_agent,
        fetch_k_per_query=fetch_k_per_query,
        final_k=final_k,
        model=model,
    )

    if chosen_agent == "regents_agent":
        answer = answer_with_regents_agent(user_query, retrieval["top_evidence"], model=model)
    elif chosen_agent == "curriculum_agent":
        answer = answer_with_curriculum_agent(user_query, retrieval["top_evidence"], model=model)
    else:
        answer = answer_with_unimplemented_agent(user_query)

    result = {
        "query": user_query,
        "route": route_info,
        "agent": chosen_agent,
        "rewrites": retrieval["rewrites"],
        "top_evidence": retrieval["top_evidence"],
        "answer": answer,
    }

    print("=" * 110)
    print("USER QUERY:", user_query)
    print("ROUTED TO:", chosen_agent)
    print("RATIONALE:", route_info.get("rationale"))
    print("CONFIDENCE:", route_info.get("confidence"))
    print("=" * 110)
    print()
    print(answer)

    if show_debug:
        print("\n--- Rewrites ---")
        for i, rq in enumerate(retrieval["rewrites"], start=1):
            print(f"{i}. {rq}")

        print("\n--- Top Evidence ---")
        for e in retrieval["top_evidence"]:
            snippet = (e.get("text", "")[:220] or "").replace("\n", " ")
            print(
                f"- {e['doc_id']} | idx={e.get('retrieved_from')} | "
                f"page={e.get('page')} | score={e.get('score', 0):.3f}"
            )
            print("  ", snippet, "...")
    return result

In [12]:
# Cell 12: examples
example_queries = [
    "Give me a Reagents practice question on solving a systems of equations. Then provide the answer step by step.",
    "My teacher taught symbolism in English class but I did not understand it. Can you try explaining it to me?",
    "Can you give me a Chemistry question about balancing a chemical reaction that is similiar in difficulty to a Reagents Exam?",
    "Teach me how to use the Pythagorean theorem."
]

for q in example_queries:
    _ = ask_multi_agent(q, routing_mode="llm", show_debug=True)
    print("\n" + "-" * 110 + "\n")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## Peter's testing version since he is a broke loser that hasnt paid for OPENAI API ... so sad

In [3]:
!pip -q install openai faiss-cpu sentence-transformers pypdf numpy transformers accelerate torch

In [24]:
# Cell 3: config
import os

ROOT_DIR = "/content/drive/MyDrive/Advanced AI/Midterm/midterm_data_clean"
INDEX_DIR = "/content/drive/MyDrive/Advanced AI/Midterm/indexes_hs_teacher_clean"

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Choose backend: "openai" or "hf"
LLM_BACKEND = "hf"

# OpenAI model if using API
OPENAI_MODEL = "gpt-5"

# Hugging Face model if testing locally
HF_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
# Alternative:
# HF_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

FETCH_K = 50
FINAL_K = 6

print("LLM_BACKEND:", LLM_BACKEND)
print("HF_MODEL_NAME:", HF_MODEL_NAME)

LLM_BACKEND: hf
HF_MODEL_NAME: HuggingFaceTB/SmolLM2-1.7B-Instruct


In [25]:
# Cell 4: imports + clients
import os, re, json, textwrap
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

st_model = SentenceTransformer(EMBED_MODEL_NAME)

client = None
hf_tokenizer = None
hf_model = None

if LLM_BACKEND == "openai":
    from openai import OpenAI
    client = OpenAI()

elif LLM_BACKEND == "hf":
    hf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)

    hf_model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL_NAME,
        torch_dtype="auto",
        device_map="auto"
    )

    # Some tokenizers do not define a pad token
    if hf_tokenizer.pad_token is None:
        hf_tokenizer.pad_token = hf_tokenizer.eos_token

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [26]:
# New cell: unified LLM wrapper
def llm_complete(
    prompt: str,
    model: Optional[str] = None,
    max_new_tokens: int = 350,
    temperature: float = 0.2,
) -> str:
    if LLM_BACKEND == "openai":
        resp = client.responses.create(
            model=model or OPENAI_MODEL,
            input=prompt,
            reasoning={"effort": "low"},
        )
        return resp.output_text.strip()

    elif LLM_BACKEND == "hf":
        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        try:
            text = hf_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            # fallback if chat template is unavailable
            text = prompt

        inputs = hf_tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        ).to(hf_model.device)

        with torch.no_grad():
            outputs = hf_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=(temperature > 0),
                temperature=temperature if temperature > 0 else None,
                pad_token_id=hf_tokenizer.pad_token_id,
                eos_token_id=hf_tokenizer.eos_token_id,
            )

        generated = outputs[0][inputs["input_ids"].shape[1]:]
        out_text = hf_tokenizer.decode(generated, skip_special_tokens=True)
        return out_text.strip()

    else:
        raise ValueError(f"Unsupported backend: {LLM_BACKEND}")

In [27]:
# Cell 5: load indexes
def load_index(index_dir: str, name: str):
    index_path = os.path.join(index_dir, f"{name}.faiss")
    meta_path = os.path.join(index_dir, f"{name}.meta.jsonl")

    if not os.path.exists(index_path):
        raise FileNotFoundError(f"Missing index file: {index_path}")
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"Missing metadata file: {meta_path}")

    index = faiss.read_index(index_path)
    meta = []
    with open(meta_path, "r", encoding="utf-8") as f:
        for line in f:
            meta.append(json.loads(line))
    return index, meta

INDEX_NAMES = [
    "standards_core",
    "curriculum_overview",
    "exam_questions",
    "exam_scoring",
]

INDEXES = {name: load_index(INDEX_DIR, name) for name in INDEX_NAMES}

for name, (idx, meta) in INDEXES.items():
    print(name, idx.ntotal, len(meta))

#print("\n===== INDEX DEBUG =====")
#for name, (idx, meta) in INDEXES.items():
#    print(
#        f"INDEX: {name} | vectors={idx.ntotal} | metadata={len(meta)}"
#    )
#print("=======================\n")

standards_core 0 0
curriculum_overview 4506 4506
exam_questions 1676 1676
exam_scoring 3147 3147


In [28]:
# Cell 6: embedding + retrieval helpers
def embed_384(texts: List[str]) -> np.ndarray:
    vecs = st_model.encode(texts, normalize_embeddings=True)
    return np.asarray(vecs, dtype=np.float32)

def infer_subject_from_query(query: str) -> Optional[str]:
    q = query.lower()
    mapping = {
        "algebra i": "Algebra 1",
        "algebra 1": "Algebra 1",
        "algebra ii": "Algebra 2",
        "algebra 2": "Algebra 2",
        "geometry": "Geometry",
        "ela": "High School English Language Arts",
        "english language arts": "High School English Language Arts",
        "english": "High School English Language Arts",
        "chemistry": "Chemistry",
        "physics": "Physics",
        "living environment": "Living Environment",
        "biology": "Life Science: Biology",
        "global history": "Global History and Geography II",
    }
    for k, v in mapping.items():
        if k in q:
            return v
    return None

def infer_admin_from_query(query: str) -> Optional[str]:
    q = query.lower()
    patterns = [
        (r"january\s+20\d{2}", None),
        (r"august\s+20\d{2}", None),
        (r"june\s+20\d{2}", None),
    ]
    for pat, _ in patterns:
        m = re.search(pat, q)
        if m:
            return m.group(0).title()
    return None

QUESTIONISH_PATTERNS = [
    r"\bPart I\b",
    r"\bQuestion\b",
    r"\([0-9]+\)",
    r"\b[A-D]\)",
    r"(?m)^\s*[0-9]{1,2}[\.)]\s",
]

def looks_like_question_text(text: str) -> bool:
    t = (text or "").replace("\n", " ")
    return any(re.search(p, t) for p in QUESTIONISH_PATTERNS)

def is_front_matter_exam(row: Dict[str, Any]) -> bool:
    src = os.path.basename(row.get("source", "")).lower()
    page = row.get("page")
    if "exam" in src and isinstance(page, int) and page <= 1:
        return not looks_like_question_text(row.get("text", ""))
    return False

def module_hint(query: str) -> Optional[str]:
    m = re.search(r"\bmodule\s*(\d+)\b", query.lower())
    return m.group(1) if m else None

def retrieve_faiss(query: str, index_name: str, fetch_k: int = FETCH_K, final_k: int = FINAL_K) -> List[Dict[str, Any]]:
    index, meta = INDEXES[index_name]
    qv = embed_384([query])
    fetch_k = min(fetch_k, len(meta))

    #print("\n--- FAISS SEARCH DEBUG ---")
    #print("Index name:", index_name)
    #print("Index vectors:", index.ntotal)
    #print("Metadata size:", len(meta))
    #print("Requested k:", fetch_k)
    #print("Query:", query)
    #print("---------------------------\n")

    #if fetch_k <= 0:
    #    print("⚠️ WARNING: k=0 so skipping FAISS search")
    #    return []

    scores, ids = index.search(qv, fetch_k)

    subj = infer_subject_from_query(query)
    admin = infer_admin_from_query(query)
    mod = module_hint(query)

    cands = []
    for sc, idx in zip(scores[0].tolist(), ids[0].tolist()):
        if idx < 0 or idx >= len(meta):
            continue
        row = dict(meta[idx])
        row["score"] = float(sc)
        cands.append(row)

    def boosted_score(r: Dict[str, Any]) -> float:
        score = r["score"]
        if subj and r.get("subject") == subj:
            score += 0.15
        if admin and r.get("admin") == admin:
            score += 0.10
        if index_name == "curriculum_overview" and mod:
            src = (r.get("source") or "").lower().replace("\\", "/")
            base = os.path.basename(src)
            if f"-m{mod}-" in base or f"/module {mod}/" in src:
                score += 0.25
        if index_name == "exam_questions" and r.get("doc_type") == "exam":
            score += 0.10
        return score

    cands.sort(key=boosted_score, reverse=True)

    if index_name == "exam_questions":
        cands = [r for r in cands if not is_front_matter_exam(r)]
        cands.sort(
            key=lambda r: (looks_like_question_text(r.get("text", "")), boosted_score(r)),
            reverse=True,
        )

    out = []
    seen = set()
    for r in cands:
        if r["doc_id"] in seen:
            continue
        seen.add(r["doc_id"])
        out.append(r)
        if len(out) >= final_k:
            break
    return out

In [29]:
# Cell 7: GPT retrieval helpers (from the gptv4-2 style)
def rewrite_queries(user_query: str, agent_name: str, index_names: List[str], model: str = OPENAI_MODEL) -> List[str]:
    prompt = f'''
Rewrite the user request into 4 short search queries for PDF retrieval.

Agent: {agent_name}
Candidate indexes: {", ".join(index_names)}

Rules:
- Return exactly 4 lines
- No numbering
- Each line <= 14 words
- Preserve subject / grade / exam / module details when present
- Prefer retrieval-oriented wording

User request: {user_query}
'''.strip()

    text = llm_complete(prompt, model=model, max_new_tokens=120, temperature=0.0)
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    if len(lines) >= 4:
        return lines[:4]
    return [user_query] * 4

def rerank_with_gpt(user_query: str, candidates: List[Dict[str, Any]], top_n: int = FINAL_K, model: str = OPENAI_MODEL) -> List[Dict[str, Any]]:
    packed = []
    for c in candidates[:60]:
        packed.append({
            "doc_id": c.get("doc_id"),
            "pdf": os.path.basename(c.get("source", "")),
            "page": c.get("page"),
            "subject": c.get("subject"),
            "admin": c.get("admin"),
            "snippet": (c.get("text", "")[:280] or "").replace("\n", " "),
        })

    prompt = f'''
You are selecting the best evidence chunks for answering a teacher-assistant query.

Return ONLY a JSON array of doc_id strings from best to worst.
Choose at most {top_n} ids.

User query:
{user_query}

Candidates:
{json.dumps(packed, ensure_ascii=False)}
'''.strip()

    text = llm_complete(prompt, model=model, max_new_tokens=180, temperature=0.0)

    try:
        chosen_ids = json.loads(text)
        chosen_ids = [x for x in chosen_ids if isinstance(x, str)]
    except Exception:
        chosen_ids = []

    by_id = {c["doc_id"]: c for c in candidates}
    reranked = [by_id[d] for d in chosen_ids if d in by_id]
    return reranked[:top_n] if reranked else candidates[:top_n]

In [30]:
# Cell 8: agent specifications
AGENT_SPECS = {
    "regents_agent": {
        "description": "Handles Regents-style practice questions, exam questions, scoring guides, rubrics, and answer explanations.",
        "indexes": ["exam_questions", "exam_scoring"],
    },
    "curriculum_agent": {
        "description": "Handles NYS curriculum teaching support, standards alignment, module guidance, lesson framing, and teaching explanations.",
        "indexes": ["curriculum_overview"],
    },
    "study_skills_agent": {
        "description": "Handles time management, study habits, and college readiness. Not implemented in this notebook.",
        "indexes": [],
    },
}

def keyword_router_fallback(query: str) -> str:
    q = query.lower()

    regents_terms = [
        "regents", "practice question", "practice problems", "multiple choice",
        "constructed response", "rubric", "rating guide", "full credit",
        "scoring", "model response", "exam"
    ]
    curriculum_terms = [
        "standard", "standards", "curriculum", "module", "lesson",
        "teach", "explain", "how do i teach", "nys", "next generation",
        "ela", "algebra", "geometry"
    ]
    study_terms = [
        "time management", "study plan", "organization", "planner",
        "college essay", "college application", "study skills"
    ]

    if any(t in q for t in regents_terms):
        return "regents_agent"
    if any(t in q for t in study_terms):
        return "study_skills_agent"
    if any(t in q for t in curriculum_terms):
        return "curriculum_agent"
    return "curriculum_agent"

def route_with_llm(query: str, model: str = OPENAI_MODEL) -> Dict[str, Any]:
    prompt = f'''
Route the user query to exactly one agent.

Available agents:
- regents_agent: Regents-style practice questions, exam questions, scoring guides, rubrics, answer explanations
- curriculum_agent: NYS curriculum teaching, standards, modules, lessons, concept explanations
- study_skills_agent: time management, study skills, college readiness (not implemented)

Return ONLY valid JSON with keys:
- agent
- rationale
- confidence

User query:
{query}
'''.strip()

    try:
        text = llm_complete(prompt, model=model, max_new_tokens=120, temperature=0.0)
        data = json.loads(text)
        if data.get("agent") not in AGENT_SPECS:
            raise ValueError("Bad agent")
        return data
    except Exception:
        fallback = keyword_router_fallback(query)
        return {
            "agent": fallback,
            "rationale": "Fallback keyword router used because LLM router output was unavailable.",
            "confidence": 0.4,
        }

In [31]:
# Cell 9: agent retrieval pipeline
def gather_agent_evidence(
    user_query: str,
    agent_name: str,
    fetch_k_per_query: int = FETCH_K,
    final_k: int = FINAL_K,
    model: str = OPENAI_MODEL,
) -> Dict[str, Any]:
    spec = AGENT_SPECS[agent_name]
    index_names = spec["indexes"]

    if not index_names:
        return {
            "rewrites": [],
            "candidate_pool": [],
            "top_evidence": [],
        }

    rewrites = rewrite_queries(user_query, agent_name, index_names, model=model)

    pool = []
    seen = set()

    # search across all indexes assigned to the agent
    per_index_final = max(final_k, 4)
    for rq in rewrites:
        for idx_name in index_names:
            hits = retrieve_faiss(
                rq,
                index_name=idx_name,
                fetch_k=fetch_k_per_query,
                final_k=per_index_final,
            )
            for h in hits:
                if h["doc_id"] not in seen:
                    seen.add(h["doc_id"])
                    h["retrieved_from"] = idx_name
                    pool.append(h)

    # lightweight score sort before GPT rerank
    pool.sort(key=lambda x: x.get("score", 0.0), reverse=True)
    pool = pool[: min(len(pool), max(25, final_k * 5))]

    top = rerank_with_gpt(user_query, pool, top_n=final_k, model=model) if pool else []

    return {
        "rewrites": rewrites,
        "candidate_pool": pool,
        "top_evidence": top,
    }

In [32]:
# Cell 10: answer-generation prompts for each implemented agent
def format_context(evidence: List[Dict[str, Any]]) -> str:
    if not evidence:
        return "NO EVIDENCE FOUND."
    chunks = []
    for i, e in enumerate(evidence, start=1):
        chunks.append(
            f'''CHUNK {i}
DOC_ID: {e["doc_id"]}
INDEX: {e.get("retrieved_from")}
PDF: {os.path.basename(e.get("source", ""))}
PAGE: {e.get("page")}
SUBJECT: {e.get("subject")}
ADMIN: {e.get("admin")}
TEXT:
{e.get("text", "")}'''
        )
    return "\n\n".join(chunks)

def answer_with_regents_agent(user_query: str, evidence: List[Dict[str, Any]], model: str = OPENAI_MODEL) -> str:
    context = format_context(evidence)
    prompt = f'''
You are the Regents Agent for a high school teacher assistant.

Your job:
- Answer Regents-related questions
- Create Regents-style practice only when the user asks for practice
- Use ONLY the evidence when making factual claims about real NYS exams, scoring guides, or released materials
- If the evidence is insufficient, say so clearly
- Never claim a generated question is an official released Regents question unless the evidence shows that

Output style:
- Clear teacher-friendly answer
- Use short sections or bullets when helpful
- End with a brief "Sources used" line listing DOC_IDs

User request:
{user_query}

Evidence:
{context}
'''.strip()

    return llm_complete(prompt, model=model, max_new_tokens=500, temperature=0.2)

def answer_with_curriculum_agent(user_query: str, evidence: List[Dict[str, Any]], model: str = OPENAI_MODEL) -> str:
    context = format_context(evidence)
    prompt = f'''
You are the Curriculum Agent for a high school teacher assistant.

Your job:
- Help teach using NYS math / ELA curriculum and standards
- Use ONLY the evidence for factual claims about curriculum structure, modules, standards, or official materials
- If the user asks for a teaching explanation, you may explain in your own words, but keep it aligned to the evidence
- If evidence is weak or incomplete, say what you can and what is missing

Output style:
- Practical and teacher-friendly
- Prioritize teaching steps, alignment, lesson framing, and student-facing explanation when relevant
- End with a brief "Sources used" line listing DOC_IDs

User request:
{user_query}

Evidence:
{context}
'''.strip()

    return llm_complete(prompt, model=model, max_new_tokens=500, temperature=0.2)

def answer_with_unimplemented_agent(user_query: str) -> str:
    return (
        "This query routed to the study-skills / time-management agent, "
        "but that agent is not implemented in this notebook yet."
    )

In [33]:
# Cell 11: orchestration function
def ask_multi_agent(
    user_query: str,
    routing_mode: str = "llm",   # options: "llm" or "manual"
    forced_agent: Optional[str] = None,
    fetch_k_per_query: int = FETCH_K,
    final_k: int = FINAL_K,
    model: str = OPENAI_MODEL,
    show_debug: bool = False,
) -> Dict[str, Any]:
    if forced_agent is not None:
        chosen_agent = forced_agent
        route_info = {
            "agent": forced_agent,
            "rationale": "Agent was manually forced by the caller.",
            "confidence": 1.0,
        }
    elif routing_mode == "manual":
        raise ValueError("For manual mode, pass forced_agent explicitly.")
    else:
        route_info = route_with_llm(user_query, model=model)
        chosen_agent = route_info["agent"]

    retrieval = gather_agent_evidence(
        user_query=user_query,
        agent_name=chosen_agent,
        fetch_k_per_query=fetch_k_per_query,
        final_k=final_k,
        model=model,
    )

    if chosen_agent == "regents_agent":
        answer = answer_with_regents_agent(user_query, retrieval["top_evidence"], model=model)
    elif chosen_agent == "curriculum_agent":
        answer = answer_with_curriculum_agent(user_query, retrieval["top_evidence"], model=model)
    else:
        answer = answer_with_unimplemented_agent(user_query)

    result = {
        "query": user_query,
        "route": route_info,
        "agent": chosen_agent,
        "rewrites": retrieval["rewrites"],
        "top_evidence": retrieval["top_evidence"],
        "answer": answer,
    }

    print("=" * 110)
    print("USER QUERY:", user_query)
    print("ROUTED TO:", chosen_agent)
    print("RATIONALE:", route_info.get("rationale"))
    print("CONFIDENCE:", route_info.get("confidence"))
    print("=" * 110)
    print()
    print(answer)

    if show_debug:
        print("\n--- Rewrites ---")
        for i, rq in enumerate(retrieval["rewrites"], start=1):
            print(f"{i}. {rq}")

        print("\n--- Top Evidence ---")
        for e in retrieval["top_evidence"]:
            snippet = (e.get("text", "")[:220] or "").replace("\n", " ")
            print(
                f"- {e['doc_id']} | idx={e.get('retrieved_from')} | "
                f"page={e.get('page')} | score={e.get('score', 0):.3f}"
            )
            print("  ", snippet, "...")
    return result

In [36]:
# Cell 12: examples
example_queries = [
    "Give me a Reagents practice question on solving a systems of equations. Then provide the answer step by step.",
    "My teacher taught symbolism in English class but I did not understand it. Can you try explaining it to me?",
    "Can you give me a Chemistry question about balancing a chemical reaction that is similiar in difficulty to a Reagents Exam?",
    "Teach me how to use the Pythagorean theorem."
]

for q in example_queries:
    #print("\n\n====================================")
    #print("RUNNING QUERY:", q)
    #print("====================================\n")

    _ = ask_multi_agent(q, routing_mode="llm", show_debug=True)

USER QUERY: Give me a Reagents practice question on solving a systems of equations. Then provide the answer step by step.
ROUTED TO: regents_agent
RATIONALE: Fallback keyword router used because LLM router output was unavailable.
CONFIDENCE: 0.4

Based on the provided evidence, the correct answer to the question "Solve the equation 1
6 (4x 1 12) 5 9 algebraically" is:

25 Solve the equation 1
6 (4x 1 12) 5 9 algebraically.

Step 1: Distribute the 1/6 to the terms inside the parentheses.
1
6 (4x 1 12) 5 9
4x 1 12 5 3

Step 2: Combine like terms.
4x 1 12 5 3
4x 1 9

Step 3: Isolate the variable x.
4x 1 9 5 0
4x 5 9
x 5 9/4

Step 4: Write the final answer.
x 5 9/4

Sources used:
- CHUNK 1: Model responses.pdf, page 68, exam_scoring
- CHUNK 2: Model responses.pdf, page 70, exam_scoring
- CHUNK 3: Model responses.pdf, page 69, exam_scoring
- CHUNK 4: Model responses.pdf, page 5, exam_scoring
- CHUNK 5: Model responses.pdf, page 1, exam_scoring
- CHUNK 6: Model responses.pdf, page 2, exam_sc

In [40]:
# Cell 13: manual / forced-agent examples
_ = ask_multi_agent(
    "Give me a Reagents practice question on solving a systems of equations. Then provide the answer step by step.",
    forced_agent="regents_agent",
    show_debug=True,
)

print("\n" + "-" * 110 + "\n")

_ = ask_multi_agent(
    "My teacher taught symbolism in English class but I did not understand it. Can you try explaining it to me?",
    forced_agent="curriculum_agent",
    show_debug=True,
)

USER QUERY: Give me a Reagents practice question on solving a systems of equations. Then provide the answer step by step.
ROUTED TO: regents_agent
RATIONALE: Agent was manually forced by the caller.
CONFIDENCE: 1.0

Based on the provided evidence, the correct solution to the equation 1
6 (4x 1 12) 5 9 is:

1
6 (4x 1 12) 5 9

Step 1: Distribute the 1
1
6 (4x 1 12) 5 9
4x 1 12 5 9

Step 2: Subtract 12 from both sides
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4x 1 12 5 9
4

--- Rewrites ---
1. 1. What are the steps to solve a system of equations using the elimination method?
2. 2. How can you solve a system of equations using the substitution method?
3.